# `dlmrel` — relation-head search on Colab

Runs the head search for **DiffuGPT-S** (smoke test, minutes) and then
**DiffuLLaMA-7B** (the real run, ~15–30 min on an A100).

**Run the cells in order and stop at the two gates.** They exist because both
failure modes this pipeline has hit produce plausible-looking numbers instead of
errors: non-eager attention returns no weights, and one checkpoint is stored under
a wrapper namespace so loading it randomly initializes the whole network.

Section 1 ends with a **runtime restart** — that is expected, not a failure.


## 1 · Setup (ends in a restart)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


Upload `dlmrel-colab.zip` (from `~/dlmresearch/`). Skip this cell if you already
dragged it into the file browser.


In [ ]:
import os
if not os.path.exists('/content/dlmrel-colab.zip'):
    from google.colab import files
    files.upload()


In [ ]:
!unzip -q -o /content/dlmrel-colab.zip -d /content/dlmrel
%cd /content/dlmrel
!ls


`transformers` **must** be 4.44.2. DiffuLLaMA's `attention_patch.py` replaces
`LlamaModel.forward` and `GPT2Model.forward` wholesale with 4.44-era
implementations and patches `LlamaFlashAttention2`, a class later releases removed.
There is no forward-compatible workaround.

`--no-deps` on the package install stops it pulling a newer transformers back in.


In [ ]:
!pip install -q 'transformers==4.44.2' 'tokenizers<0.20' 'huggingface-hub<0.37' \
               accelerate sentencepiece conllu
!pip install -q -e . --no-deps
print('\n>>> Now do Runtime -> Restart session, then continue at section 2. <<<')


## 2 · After the restart

Everything below assumes the runtime has been restarted.


In [ ]:
%cd /content/dlmrel
from google.colab import drive; drive.mount('/content/drive')
%env HF_HOME=/content/drive/MyDrive/hf-cache


Caching the weights on Drive matters: DiffuLLaMA-7B is ~13.5 GB, and a
disconnect without this costs more wall-clock than the experiments themselves.


In [ ]:
import sys, transformers
print('transformers', transformers.__version__)
assert transformers.__version__ == '4.44.2', 'wrong version — restart the runtime'

# Importing DiffuLLaMA's model.py applies the attention patch. If this raises,
# the version pin did not take.
!git clone -q --depth 1 https://github.com/HKUNLP/DiffuLLaMA.git third_party/DiffuLLaMA 2>/dev/null || true
sys.path.insert(0, '/content/dlmrel/third_party/DiffuLLaMA')
import model as _diffullama
print('attention patch applied OK')


In [ ]:
!python -m pytest -q


## 3 · Gate 1 — is the model real?

Every GPT-2-derived model has previous-token and next-token heads carrying ~0.9 of
their attention mass. If they are absent, we are not reading real attention and
nothing downstream means anything.


In [ ]:
!dlmrel data --config configs/diffugpt-s.yaml


In [ ]:
!python scripts/diagnose_heads.py --config configs/diffugpt-s.yaml --n 200


**Read the output before continuing.**

| check | expected | if it fails |
|---|---|---|
| best previous-token head | ~0.85+ | weights did not load — check for `remapped 148 tensors` above |
| best next-token head | clearly > 0 | causal mask not removed; the model is running autoregressively |
| object→verb peak offset | **0** | non-zero means the word-to-token alignment is shifted |

A next-token head at exactly 0.000 is the one to watch for: it means
`gpt2block.attn.bias.fill_(True)` did not take effect.


## 4 · Gate 2 — does DiffuGPT-S reproduce?

Known answer: **L4 H0 ≈ 0.75** on object→verb. A few minutes on any GPU.


In [ ]:
!dlmrel search  --config configs/diffugpt-s.yaml
!dlmrel analyze --config configs/diffugpt-s.yaml


In [ ]:
import pandas as pd
t = pd.read_csv('results/diffugpt-s-ewt/head_vs_null.csv').set_index('relation')
r = t.loc['object_to_verb']
print(f"object->verb: L{int(r.layer)} H{int(r['head'])} = {r.head_test_acc:.3f} "
      f"(null {r.null_test_acc:.3f}, {r.n_heads_above_null}/{r.n_heads_total} heads above, "
      f"rho={r.selection_rho:.2f})")

ok = r.head_test_acc > 0.60 and r.n_heads_above_null > 10 and r.selection_rho > 0.9
print('\nGATE:', 'PASS — go on to the 7B' if ok else 'FAIL — do not start the 7B')
if not ok:
    print('  Chance is ~0.06. Near-chance accuracy, 0 heads above null, or rho < 0.9')
    print('  all point at the pipeline rather than the science.')


## 5 · The real run — DiffuLLaMA-7B

1024 heads, ~6,000 forward passes, ~15–30 min. `analyze` prints the head-vs-null
table plus per-head `selectivity` and `adjacency_bias` — that is where the
positional/relational dissociation shows up, if it is there.

Note `model.safetensors` is ~13.5 GB on first run; subsequent sessions read it from Drive.


In [ ]:
!dlmrel data    --config configs/default.yaml


In [ ]:
!dlmrel search  --config configs/default.yaml


In [ ]:
!dlmrel analyze --config configs/default.yaml


## 6 · Save before the session dies

`results/` is on ephemeral disk.


In [ ]:
!mkdir -p /content/drive/MyDrive/dlmrel-results
!cp -r results/* /content/drive/MyDrive/dlmrel-results/
!ls -R /content/drive/MyDrive/dlmrel-results | head -40


## 7 · Summary to paste back


In [ ]:
import pandas as pd
for name, path in [('DiffuGPT-S (144 heads)', 'results/diffugpt-s-ewt'),
                   ('DiffuLLaMA-7B (1024 heads)', 'results/diffullama-ewt')]:
    try:
        t = pd.read_csv(f'{path}/head_vs_null.csv')
    except FileNotFoundError:
        continue
    cols = ['relation','layer','head','head_test_acc','head_ci_lo','null_test_acc',
            'delta','n_heads_above_null','selection_rho','verdict']
    print(f'\n=== {name} ===')
    print(t[cols].to_string(index=False))


In [ ]:
# Split overlap between the two models: the tokenizers admit slightly different
# sentence pools, so the two arms are not scored on identical sentences.
import pandas as pd
a = set(pd.read_csv('results/diffugpt-s-ewt/sentences_test.csv')['sentence'])
b = set(pd.read_csv('results/diffullama-ewt/sentences_test.csv')['sentence'])
print(f'test sentences: {len(a)} vs {len(b)}, overlap {len(a & b)} ({len(a & b)/len(a):.1%})')
print('>95% is a footnote; lower means we restrict both models to the intersection.')


## Not yet — `dlmrel curve`

As configured that is 5 seeds x 64 timesteps x 1000 sentences = **320,000 forward
passes**, roughly 13 hours. It needs timestep-stride and sentence-count knobs first
so the trimmed ~1.8 h version is a config change. Run `search` and `analyze` first
and look at the head profiles.
